# Runbook 3.4.a: Firewall types (stateful vs. stateless)  (MODEL SOLUTION, instructor copy, do not distribute)

**Skill:** Mitigate Risk · **Maps to:** 3.4.A · **Language:** Python 3 (pytest)

**Steps:** 1 Brief · 2 Plan · 3 Build · 4 Test · 5 Try it on the evidence · 6 Wrap up

## 1. Brief

**The situation.** You will implement two firewall decision functions that together illustrate the core difference
between stateless and stateful packet filtering.

A **packet** is a dict with these string keys:
- `"src"`: source IP string (e.g. `"203.0.113.10"`)
- `"dst"`: destination IP string
- `"sport"`: source port integer
- `"dport"`: destination port integer
- `"proto"`: protocol string, one of `"tcp"` / `"udp"` / `"icmp"`
- `"flags"`: TCP flag string: one of `"SYN"`, `"SYN-ACK"`, `"ACK"`, `"RST"`, `"FIN"`, `"DATA"`
  (non-TCP packets use `""`)

A **rule** is a dict with:
- `"action"`, `"allow"` or `"drop"`
- `"proto"`: protocol to match (`"tcp"`, `"udp"`, `"icmp"`, or `"any"`)
- `"dport"`: destination port to match (integer) or `0` meaning any port

Rules are evaluated top-to-bottom; the first match decides.  If no rule matches, the implicit
verdict is `"drop"`.

**Lesson objective.** Distinguish stateless from stateful firewalls, explain how each inspects traffic, and identify which type is better suited to a given threat scenario.

### What comes in
```python
rules = [
    {"action": "allow", "proto": "tcp", "dport": 443},
    {"action": "allow", "proto": "tcp", "dport": 80},
    {"action": "drop",  "proto": "any", "dport": 0},
]
packets = [
    {"src": "203.0.113.10", "dst": "10.0.0.5", "sport": 55001, "dport": 443, "proto": "tcp", "flags": "SYN"},
    {"src": "10.0.0.5",     "dst": "203.0.113.10", "sport": 443, "dport": 55001, "proto": "tcp", "flags": "SYN-ACK"},
    {"src": "198.51.100.7", "dst": "10.0.0.5", "sport": 44000, "dport": 443, "proto": "tcp", "flags": "ACK"},
]
```

### What must come out
```
stateless: ["allow", "drop", "allow"]   # SYN-ACK (dport 55001) matches no rule
stateful:  ["allow", "allow", "drop"]   # third packet: ACK with no prior SYN
```

- [ ] Read the situation and say in one sentence what the tool must decide
- [ ] Find the input and the output above

## 2. Plan

Work it out on paper before you write code.


### Rules to respect
- [ ] Deterministic, offline, standard library only.
- [ ] Must pass all cases in the tests (visible + hidden).
- [ ] The connection table is local to each `solve()` call (not persisted between calls).
- [ ] Write the steps your function will take, in plain words
- [ ] List two edge cases (an empty input, a tricky value)

<details><summary>Reference: the exact contract</summary>

Any implementation that satisfies this interface can be swapped in (reference, student, or
composite) and must pass the tests unchanged.

#### Entry point
```python
def solve(rules: list[dict], packets: list[dict], mode: str) -> list[str]
```

#### Input

##### `rules`, list of rule dicts, evaluated top-to-bottom (first match wins)
Each rule dict has:
- `"action"` (str): `"allow"` or `"drop"`
- `"proto"` (str): `"tcp"`, `"udp"`, `"icmp"`, or `"any"`
- `"dport"` (int): destination port to match, or `0` meaning any port

##### `packets`, list of packet dicts, processed in order
Each packet dict has:
- `"src"` (str): source IP address
- `"dst"` (str): destination IP address
- `"sport"` (int): source port
- `"dport"` (int): destination port
- `"proto"` (str): `"tcp"`, `"udp"`, or `"icmp"`
- `"flags"` (str): TCP flag, one of `"SYN"`, `"SYN-ACK"`, `"ACK"`, `"RST"`, `"FIN"`, `"DATA"`,
  or `""` for non-TCP

##### `mode`, `"stateless"` or `"stateful"`

#### Output (`result`)
`list[str]` of verdicts, one per packet, each `"allow"` or `"drop"`, in the same order as
`packets`.

#### Rule matching (both modes)
A rule matches a packet when BOTH conditions hold:
1. `rule["proto"] == "any"` OR `rule["proto"] == packet["proto"]`
2. `rule["dport"] == 0` OR `rule["dport"] == packet["dport"]`
Implicit default if no rule matches: `"drop"`.

#### Stateful TCP tracking (stateful mode only)
- Connection table key: `(src, dst, sport, dport, proto)`, all five fields.
- **SYN**: passes rules -> add key with state `"SYN_SENT"` -> `"allow"`.
- **SYN-ACK**: reverse key `(dst, src, dport, sport, proto)` in table -> update to
  `"ESTABLISHED"` -> `"allow"`. Reverse key not found -> `"drop"`.
- **ACK / DATA**: key or reverse in table with state `"ESTABLISHED"` -> `"allow"`.
  Otherwise -> `"drop"` (no established session for this endpoint pair).
- **RST / FIN**: key or reverse in table -> remove entry -> `"allow"`. Not found -> `"drop"`.
- Non-TCP packets (UDP, ICMP): rule matching only; no state tracking.

#### Guarantees / rules
- Pure function; deterministic; no I/O; no network access.
- Connection table is initialized empty at the start of each `solve()` call.
- Standard library only.

#### Acceptance
Your code passes when every test below shows a checkmark (some are hidden until you run them).

</details>

## 3. Build it

This is the model solution.

- [ ] `solve` handles the worked examples
- [ ] `solve` handles your edge cases

In [ ]:
from typing import List, Dict, Any


def _match_rules(rules: List[Dict], packet: Dict) -> str:
    for rule in rules:
        proto_ok = rule["proto"] == "any" or rule["proto"] == packet["proto"]
        dport_ok = rule["dport"] == 0 or rule["dport"] == packet["dport"]
        if proto_ok and dport_ok:
            return rule["action"]
    return "drop"


def solve(rules: List[Dict], packets: List[Dict], mode: str) -> List[str]:
    if mode == "stateless":
        return [_match_rules(rules, pkt) for pkt in packets]

    table: Dict[tuple, str] = {}
    verdicts: List[str] = []

    for pkt in packets:
        proto = pkt["proto"]
        flags = pkt.get("flags", "")
        key = (pkt["src"], pkt["dst"], pkt["sport"], pkt["dport"], proto)
        rev = (pkt["dst"], pkt["src"], pkt["dport"], pkt["sport"], proto)

        if proto != "tcp":
            verdicts.append(_match_rules(rules, pkt))
            continue

        if flags == "SYN":
            rule_verdict = _match_rules(rules, pkt)
            if rule_verdict == "allow":
                table[key] = "SYN_SENT"
                verdicts.append("allow")
            else:
                verdicts.append("drop")

        elif flags == "SYN-ACK":
            if rev in table:
                table[rev] = "ESTABLISHED"
                verdicts.append("allow")
            else:
                verdicts.append("drop")

        elif flags in ("ACK", "DATA"):
            if table.get(key) == "ESTABLISHED" or table.get(rev) == "ESTABLISHED":
                verdicts.append("allow")
            else:
                verdicts.append("drop")

        elif flags in ("RST", "FIN"):
            if key in table or rev in table:
                table.pop(key, None)
                table.pop(rev, None)
                verdicts.append("allow")
            else:
                verdicts.append("drop")

        else:
            verdicts.append(_match_rules(rules, pkt))

    return verdicts


## 4. Test it

The tests are the acceptance check, like a change review before a fix goes live.

- [ ] Run the tests
- [ ] Every test says PASS (if not, read the message, go back to step 3)

In [ ]:
import __main__ as impl

def test_example_stateless():
    rules = [
        {"action": "allow", "proto": "tcp", "dport": 443},
        {"action": "allow", "proto": "tcp", "dport": 80},
        {"action": "drop", "proto": "any", "dport": 0},
    ]
    packets = [
        {"src": "203.0.113.10", "dst": "10.0.0.5", "sport": 55001, "dport": 443, "proto": "tcp", "flags": "SYN"},
        {"src": "10.0.0.5", "dst": "203.0.113.10", "sport": 443, "dport": 55001, "proto": "tcp", "flags": "SYN-ACK"},
        {"src": "198.51.100.7", "dst": "10.0.0.5", "sport": 44000, "dport": 443, "proto": "tcp", "flags": "ACK"},
    ]
    assert impl.solve(rules, packets, "stateless") == ["allow", "drop", "allow"]


def test_example_stateful():
    rules = [
        {"action": "allow", "proto": "tcp", "dport": 443},
        {"action": "allow", "proto": "tcp", "dport": 80},
        {"action": "drop", "proto": "any", "dport": 0},
    ]
    packets = [
        {"src": "203.0.113.10", "dst": "10.0.0.5", "sport": 55001, "dport": 443, "proto": "tcp", "flags": "SYN"},
        {"src": "10.0.0.5", "dst": "203.0.113.10", "sport": 443, "dport": 55001, "proto": "tcp", "flags": "SYN-ACK"},
        {"src": "198.51.100.7", "dst": "10.0.0.5", "sport": 44000, "dport": 443, "proto": "tcp", "flags": "ACK"},
    ]
    assert impl.solve(rules, packets, "stateful") == ["allow", "allow", "drop"]


def test_stateful_handshake_flow():
    rules = [
        {"action": "allow", "proto": "tcp", "dport": 80},
        {"action": "drop", "proto": "any", "dport": 0}
    ]
    packets = [
        {"src": "10.0.0.2", "dst": "192.168.1.10", "sport": 12345, "dport": 80, "proto": "tcp", "flags": "SYN"},
        {"src": "192.168.1.10", "dst": "10.0.0.2", "sport": 80, "dport": 12345, "proto": "tcp", "flags": "SYN-ACK"},
        {"src": "10.0.0.2", "dst": "192.168.1.10", "sport": 12345, "dport": 80, "proto": "tcp", "flags": "ACK"},
        {"src": "192.168.1.10", "dst": "10.0.0.2", "sport": 80, "dport": 12345, "proto": "tcp", "flags": "DATA"},
        {"src": "10.0.0.2", "dst": "192.168.1.10", "sport": 12345, "dport": 80, "proto": "tcp", "flags": "FIN"},
        {"src": "192.168.1.10", "dst": "10.0.0.2", "sport": 80, "dport": 12345, "proto": "tcp", "flags": "DATA"}
    ]
    assert impl.solve(rules, packets, "stateful") == ["allow", "allow", "allow", "allow", "allow", "drop"]


def test_non_tcp_stateful():
    rules = [
        {"action": "allow", "proto": "udp", "dport": 53},
        {"action": "drop", "proto": "any", "dport": 0}
    ]
    packets = [
        {"src": "10.0.0.2", "dst": "8.8.8.8", "sport": 12345, "dport": 53, "proto": "udp", "flags": ""},
        {"src": "10.0.0.2", "dst": "8.8.8.8", "sport": 12345, "dport": 54, "proto": "udp", "flags": ""}
    ]
    assert impl.solve(rules, packets, "stateful") == ["allow", "drop"]


def test_empty():
    assert impl.solve([], [], "stateless") == []
    assert impl.solve([], [], "stateful") == []


# ---- run every test above ------------------------------------------------
_results = []
for _name, _fn in list(globals().items()):
    if _name.startswith("test_") and callable(_fn):
        try:
            _fn()
            _results.append((_name, None))
        except Exception as _e:
            _results.append((_name, _e))
for _name, _e in _results:
    print(("PASS  " if _e is None else "FAIL  ") + _name + ("" if _e is None else f": {_e!r}"))
print(f"{sum(e is None for _, e in _results)}/{len(_results)} tests passed")


## 5. Try it on the evidence

Run your finished tool on the examples from the brief, then change one value and predict the answer before you run it again.

- [ ] Every example says OK
- [ ] Your own changed example gives the answer you predicted

In [ ]:
checks = [
    (lambda: solve([], [], 'stateless'),
     []),
    (lambda: solve([], [], 'stateful'),
     []),
]

for n, (run, want) in enumerate(checks, 1):
    try:
        got = run()
    except Exception as e:
        got = "error: " + repr(e)
    print("example", n, "OK  " if got == want else "DIFF", "got:", got, " expected:", want)


## 6. Wrap up

- [ ] Explain in one sentence why your rule catches the problem
- [ ] Name one way an attacker could slip past it (a false negative)
- [ ] Name one harmless input it might wrongly flag (a false positive)
- [ ] Say where this check would run in a real organisation, and who would act on it